In [40]:
print("This is the versione jupyther notebbok script!")

This is the versione jupyther notebbok script!


In [41]:
import sys
print(sys.executable)

/workspaces/llm-zoomcamp2026/01-agentic-rag/code/.venv/bin/python3


In [42]:
from dotenv import load_dotenv
load_dotenv()

True

In [43]:
import os
import os

api_key=os.getenv("GROQ_API_KEY")
if api_key is None:
    raise ValueError("GROQ_API_KEY environment variable not set")
else:
    print("GROQ_API_KEY is set correctly")

GROQ_API_KEY is set correctly


In [44]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [45]:
def llm_gpt(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [46]:
def llm_groq(prompt):
    response = openai_client.chat.completions.create(  # ← usa questa API
        #model="llama-3.3-70b-versatile",              # ← model ID Groq
        model="groq/compound-mini",                          # ← model ID Groq
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content         # ← risposta qui

In [47]:
llm_groq("What is the capital of France?")

'The capital of France is **Paris**.'

In [48]:
models = openai_client.models.list()
for m in models: print(m.id)

meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-120b
groq/compound-mini
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-arabic-saudi
llama-3.1-8b-instant
allam-2-7b
groq/compound
whisper-large-v3-turbo
qwen/qwen3-32b
llama-3.3-70b-versatile
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3


In [49]:
llm_groq("Chi è stato il tuo inventore?")

'Sono stato creato dal team di ingegneri e ricercatori di **Groq**.'

In [50]:
question = "I just discovered the course. Can I join now?"
answer = llm_groq(question)
print(answer)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kt4rwb05e84bvttg2gqyf6rw` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 4994, Requested 5372. Please try again in 17.745s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'compound', 'code': 'rate_limit_exceeded'}}

In [ ]:
question = "I just discovered the course. Can I join now?"
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [ ]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
print(f'Question: {question}')
answer = llm_groq(prompt)
print(f'Answer: {answer}')

Question: I just discovered the course. Can I join now?
Answer: Yes—you can still join the course. If you’d like to earn a certificate, just be sure to submit your project before the submission deadline.


In [ ]:
# La struttura di quanto realizzeremo è:def rag(question):
#1) Ricerca: data la domanda dell'utente, cerchiamo nel KB (Knowledge Base) per trovare informazioni rilevanti.
search_results = search(question)
#2) Prompting: costruiamo un prompt che includa la domanda dell'utente e i risultati della ricerca.
user_prompt = build_prompt(question, search_results)
#3) Risposta: inviamo il prompt al modello LLM per ottenere una risposta.    
return llm(user_prompt)

In [56]:
import requests
#Link con la lista dei corsi DataTalks e indicazione dei link alle faq di ciascun corso
docs_url = "https://datatalks.club/faq/json/courses.json" 
response = requests.get(docs_url)
#This returns a list of courses, each course has a path field that points to its FAQ data.
courses_raw = response.json()
print(type(courses_raw))
print(type(courses_raw[0]))
print(courses_raw[3])
url_prefix_faq = "https://datatalks.club/faq"
index = 0
for course in courses_raw:
    index = index + 1
    print('---')
    print(f'Index: {index}')
    print(f'Course id: {course["course"]}')
    print(f'Course name: {course["course_name"]}')
    print(f'Path FAQ: {course["path"]}')
    print(f'FAQ URL: {url_prefix_faq}{course["path"]}')


<class 'list'>
<class 'dict'>
{'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 79}
---
Index: 1
Course id: data-engineering-zoomcamp
Course name: Data Engineering Zoomcamp
Path FAQ: /json/data-engineering-zoomcamp.json
FAQ URL: https://datatalks.club/faq/json/data-engineering-zoomcamp.json
---
Index: 2
Course id: stock-markets-analytics-zoomcamp
Course name: Stock Markets Analytics Zoomcamp
Path FAQ: /json/stock-markets-analytics-zoomcamp.json
FAQ URL: https://datatalks.club/faq/json/stock-markets-analytics-zoomcamp.json
---
Index: 3
Course id: ai-dev-tools-zoomcamp
Course name: AI Dev Tools Zoomcamp
Path FAQ: /json/ai-dev-tools-zoomcamp.json
FAQ URL: https://datatalks.club/faq/json/ai-dev-tools-zoomcamp.json
---
Index: 4
Course id: llm-zoomcamp
Course name: LLM Zoomcamp
Path FAQ: /json/llm-zoomcamp.json
FAQ URL: https://datatalks.club/faq/json/llm-zoomcamp.json
---
Index: 5
Course id: mlops-zoomcamp
Course name: MLOps Zoom

In [57]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

print(len(documents))
documents[600]

1342


{'id': 'eae0fb50aa',
 'course': 'llm-zoomcamp',
 'section': 'Capstone Project',
 'question': 'When and how will we be assigned projects for review/grading?',
 'answer': 'After the submission deadline has passed, an Excel sheet will be shared with 3 projects being assigned to each participant.'}

In [58]:
from minsearch import Index

# Create documents
docs = [
    {
        "question": "How do I join the course after it has started?",
        "text": "You can join the course at any time. We have recordings available.",
        "section": "General Information",
        "course": "data-engineering-zoomcamp"
    },
    {
        "question": "What are the prerequisites for the course?",
        "text": "You need to have basic knowledge of programming.",
        "section": "Course Requirements",
        "course": "data-engineering-zoomcamp"
    }
]

# Create and fit the index
index = Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)
index.fit(docs)

# Search with filters and boosts
query = "Can I join the course if it has already started?"
filter_dict = {"course": "data-engineering-zoomcamp"}
boost_dict = {"question": 3, "text": 1, "section": 1}

results = index.search(query, filter_dict=filter_dict, boost_dict=boost_dict)
print(results)

[{'question': 'How do I join the course after it has started?', 'text': 'You can join the course at any time. We have recordings available.', 'section': 'General Information', 'course': 'data-engineering-zoomcamp'}, {'question': 'What are the prerequisites for the course?', 'text': 'You need to have basic knowledge of programming.', 'section': 'Course Requirements', 'course': 'data-engineering-zoomcamp'}]


In [59]:
def search(question, course="llm-zoomcamp"):
    filter_dict = {"course": course}
    boost_dict = {"question": 3, "answer": 1, "section": 1}
    num_results = 5  # Number of results to return
    results = index.search(question, filter_dict=filter_dict, boost_dict=boost_dict, num_results=num_results)
    return results

In [61]:
from minsearch import Index

docs = documents
print(f"Number of documents: {len(docs)}")
# Create and fit the index
index = Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)
index.fit(docs)
question = 'Groq LLM provider'
print(f"Searching for: {question}")
search_results = index.search(question)
#print(search_results)
search_results

Number of documents: 1342
Searching for: Groq LLM provider


[{'id': 'dec5edee6a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 1: Terraform',
  'question': 'Terraform: Could not reach provider registry in restricted region (e.g., Iraq) – Invalid provider registry host / Could not query available provider packages',
  'answer': 'This error can occur when Terraform cannot access the online provider registry, which may happen in restricted regions where registry.terraform.io is blocked. The error messages may include things like "Invalid provider registry host" and "Could not query available versions for provider hashicorp/google".\n\nFix:\n\n- Install and activate a system-wide VPN software (e.g., Proton VPN). Ensure that all traffic from your terminal (where `terraform init` runs) is routed through the VPN.\n- Important: A VPN browser extension is usually not sufficient. Extensions proxy traffic only within the web browser and do not affect terminal/VS Code connections.\n\nAfter the VPN is active, run `terraform init` again to r

In [62]:
#This is what grounds the answer in our data and reduces hallucinations.
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [63]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        #answer = doc["answer"].replace("<{IMAGE:image_1}>", "").strip()
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [64]:
prompt = build_prompt(question, search_results)

print(f"Prompt length: {len(prompt)} chars")

# Stampa solo l'inizio
print(prompt[:1000])

Prompt length: 4559 chars
Question:
Groq LLM provider

Context:
Module 1: Terraform
Q: Terraform: Could not reach provider registry in restricted region (e.g., Iraq) – Invalid provider registry host / Could not query available provider packages
A: This error can occur when Terraform cannot access the online provider registry, which may happen in restricted regions where registry.terraform.io is blocked. The error messages may include things like "Invalid provider registry host" and "Could not query available versions for provider hashicorp/google".

Fix:

- Install and activate a system-wide VPN software (e.g., Proton VPN). Ensure that all traffic from your terminal (where `terraform init` runs) is routed through the VPN.
- Important: A VPN browser extension is usually not sufficient. Extensions proxy traffic only within the web browser and do not affect terminal/VS Code connections.

After the VPN is active, run `terraform init` again to retry fetching provider packages.

General Cour

In [65]:
response = openai_client.chat.completions.create(  # ← usa questa API
    #model="llama-3.3-70b-versatile",              # ← model ID Groq
    model="groq/compound-mini",                          # ← model ID Groq
    messages=[{"role": "user", "content": prompt}]
)
print(response.choices[0].message.content)
print(type(response.choices[0].message.content))
print(response.usage)

**Short answer:** Yes – you can use Groq as the LLM provider for the Zoomcamp labs (or any of the homework notebooks). It works the same way as OpenAI, Anthropic, Gemini, etc.; you just need to swap the client‑initialisation code and, if needed, adjust a few parameters (e.g., the `response_format` or `model` name).

---

### How to switch to Groq

| Step | What to do | Example |
|------|------------|---------|
| **1. Install the Groq Python SDK** | `pip install groq` | ```bash pip install groq ``` |
| **2. Get an API key** | Sign‑up at <https://groq.com> and copy the key from the dashboard. | `export GROQ_API_KEY=your_key_here` |
| **3. Initialise the client** | Replace the OpenAI client with a Groq client. | ```python from groq import Groq  client = Groq(api_key=os.getenv("GROQ_API_KEY")) ``` |
| **4. Call the model** | Use the same chat‑completion interface; just change the model name (`llama3‑8b‑8192`, `mixtral‑8x7b‑32768`, etc.). | ```python response = client.chat.completions.creat

In [66]:
from groq import Groq
client = Groq()
completion = client.chat.completions.create(
    #model="openai/gpt-oss-120b",
    model="groq/compound-mini",
    messages=[
        {
            "role": "user",
            "content": "Explain why fast inference is critical for reasoning models"
        }
    ]
)
print(completion.choices[0].message.content)

Fast inference — the ability to generate a model’s output with very low latency—matters a lot for reasoning‑oriented language models. Here are the key reasons:

| Reason | Why it matters for reasoning models |
|--------|--------------------------------------|
| **Real‑time interaction** | Many reasoning applications (chatbots, tutoring systems, decision‑support tools, code assistants, etc.) expect the model to answer within a second or two. If the model takes long to compute a chain of logical steps, the conversation feels sluggish and users lose trust. |
| **Multi‑step reasoning latency compounds** | Reasoning models often perform several internal “thought” steps (e.g., chain‑of‑thought prompting, tool use, self‑consistency sampling). Each step adds its own inference time, so a slow base model quickly balloons into seconds or minutes of total latency. Fast inference keeps the overall response time acceptable. |
| **Scalability & throughput** | Deployments serving thousands or millions

In [67]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

completion = client.chat.completions.create(
    model="groq/compound-mini",
    messages=message_history
)
print(completion.choices[0].message.content)

**Yes – you can use Groq as your LLM provider.**  

- The course is designed to work with any LLM platform, so you’re free to replace OpenAI with Groq for your experiments and homework.  
- You’ll need to adjust the code to call Groq’s API (e.g., set the appropriate endpoint, authentication token, and use Groq’s `response_format` parameter when you need structured JSON output).  
- Refer to Groq’s official documentation for details on authentication, request format, and any SDKs they provide.  

In short, Groq works as a drop‑in alternative; just make the necessary code changes and follow Groq’s API docs.


In [68]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = client.chat.completions.create(
        model=model,
        messages=message_history
    )

    return response.choices[0].message.content

In [69]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [75]:
#question = "How many people usually follow DataTalkClub llm-course?"
question = "Do I need a GitHub account to join the course?"
answer = rag(question, model="groq/compound-mini")
print(answer)

No. You can join the course using Google or Slack – a GitHub account is not required (the GitHub login sometimes fails, so the recommended alternatives are Google or Slack).
